# CiTree Scraper

[CiTree](https://citree.de/) stellt Informationen über verschiedene Baumarten bereit. Um diese Daten zu verarbeiten, müssen wir sie einmalig herunterladen und aufbereiten. Da wir keinen Datenbank-Dump erhalten, haben wir die Daten mit diesem Notebook heruntergeladen und aufbereitet.

## Download der Baumarten

Da CiTree keine API bereitstellt, um strukturierte Daten im JSON-Format abzurufen, müssen wir diese Daten selbst aufbereiten. Hierfür laden wir im ersten Schritt die Daten herunter und legen sie lokal ab. Hierbei handelt es sich um partielle HTML-Dateien, die auf der Webseite angezeigt werden.

## Lizenzen

Die Inhalte der Website unterliegen der [Creative-Commons-Lizenz BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.de). Das beduetet, wir dürfen sie nichtkommerziell weiterverwenden, aber mit Namensnennung.

In [ ]:
from tqdm import tqdm
import requests
import os

treeId = 1

image_urls = set()

if not os.path.exists("data"):
    os.makedirs("data")

for treeId in tqdm(range(1, 375)):
    if os.path.exists(f"data/{treeId}.html"):
        continue
    URL = f"https://citree.de/AJAX/getSteckbrief.php?ArtInfos=%7B%22ID%22%3A{treeId}%2C%22matched%22%3A%5B44%5D%2C%22unmatched%22%3A%5B0%5D%7D&language=de"
    response = requests.get(URL)
    with open(f"data/{treeId}.html", "w") as f:
        f.write(response.text)

100%|██████████| 2/2 [00:01<00:00,  1.51it/s]


## Parsen der Attribute und Download von Bildern

Im ersten Schritt müssen wir die verfügbaren Attribute der Bäume ermitteln. Hierfür iterieren wir einmal über alle heruntergeladenen Baumarten und suchen alle Attribute aus den HTML-Tabellen heraus.

Außerdem werden in diesem Zuge auch die Bilder der jeweiligen Bäume in der höchsten verfügbaren Qualität heruntergeladen.

In [ ]:
from bs4 import BeautifulSoup
from tqdm import tqdm
import os

attributes = {}

if not os.path.exists("images"):
    os.makedirs("images")

for tree_id in tqdm(range(1, 366)):

    with open(f"data/{tree_id}.html", "r") as f:
        html_content = f.read()
    soup = BeautifulSoup(html_content, 'html.parser')
    
    tree_image_urls = [i.get('src') for i in soup.find_all('img') if 'img/icons/ic_close_dark.png' not in i.get('src')]
    for url in tree_image_urls:
        if os.path.exists(f"images/{url.split('/')[-1]}"):
            continue
        large_image_url = url.replace("/640/", "/1024/")
        download_url = f"https://citree.de/{large_image_url}"
        response = requests.get(download_url)
        if response.status_code == 200:
            with open(f"images/{large_image_url.split('/')[-1]}", "wb") as f:
                f.write(response.content)

    headlines = soup.find_all('h4')
    for h in headlines:
        # print(h.text.strip())

        attribute_name = h.text.strip()

        if attribute_name not in attributes:
            attributes[attribute_name] = {}

        table = h.next_sibling
        for row in table.find_all('tr'):
            if row.find('td').text.strip() not in attributes[attribute_name]:
                attributes[attribute_name][row.find('td').text.strip()] = row['title']

for a in attributes:
    print(a)
    for b in attributes[a]:
        print(f"  - {b}: {attributes[a][b]}")


100%|██████████| 2/2 [00:00<00:00, 112.29it/s]

Klimabedingungen
  - Lichtanspruch: Abschattung des Baumstandortes durch benachbarte Bäume oder Gebäude
  - Trockenheitstoleranz: Gefährdung durch Luft-und Bodentrockenheit
  - Hitzeverträglichkeit: Gefährdung durch Hitzebelastung
  - Spätfrosttoleranz: Gebiete die als spätfrostgefährdet gelten sind Senken, Flussnähe, Freiflächen
  - Winterhärtezone: Einteilung der Klimaregionen anhand der durchschnittlich kältesten Jahrestemperatur
Bodenbedingungen
  - pH-Wert: pH-Wert am Pflanzstandort
  - Bodenverdichtungstoleranz: Risiko des Auftretens von Bodenverdichtung
  - Staunässetoleranz: Risiko des Auftretens von Staunässe
  - Salzverträglichkeit: Wie hoch ist die Salzbelastung
  - Bodenfeuchtetoleranz: Bodenfeuchtigkeit am Standort
  - Bodeneigenschaft: Substrat auf Pflanzenstandort
  - Gründigkeit: Wie tief wurzelt die Baumart
Natürliche Verbreitung
  - Neophyt: Keine heimische Pflanze (wurde nach 1492 eingeführt)
  - Herkunft: 
  - Natürlicher Lebensraum: Natürlicher Lebensraum der Pflan

## Parsen der Baumdaten

Anhand der ermittelten Attribute können wir die Baumdaten nun parsen.

In [ ]:
from bs4 import BeautifulSoup
import re

trees = []

for tree_id in tqdm(range(1, 366)):
    tree_information = {}

    tree_information['id'] = tree_id
    
    trivial_name_tag = soup.find('span', class_='name-trivial')
    tree_information['name_trivial'] = trivial_name_tag.text.strip() if trivial_name_tag else None

    botanic_name_tag = soup.find('span', class_='name-botanic')
    tree_information['name_botanic'] = botanic_name_tag.text.strip() if botanic_name_tag else None

    strain_tag = soup.find('div', class_='strain')
    tree_information['strain'] = strain_tag.text.strip().replace("Sorte: ", "") if strain_tag else None

    with open(f"data/{tree_id}.html", "r") as f:
        html_content = f.read()
    soup = BeautifulSoup(html_content, 'html.parser')
    tree_information['id'] = tree_id
    
    trivial_name_tag = soup.find('span', class_='name-trivial')
    tree_information['name_trivial'] = trivial_name_tag.text.strip() if trivial_name_tag else None

    botanic_name_tag = soup.find('span', class_='name-botanic')
    tree_information['name_botanic'] = botanic_name_tag.text.strip() if botanic_name_tag else None

    strain_tag = soup.find('div', class_='strain')
    tree_information['strain'] = strain_tag.text.strip().replace("Sorte: ", "").strip() if strain_tag else None

    tree_image_urls = [a.get('href').replace("img/1024/", "") for a in [i.parent for i in soup.find_all('img', class_="img-responsive") if 'img/icons/ic_close_dark.png' not in i.get('src')]]
    tree_information['images'] = tree_image_urls

    for attribute_group in attributes:
        for attribute in attributes[attribute_group]:

            value = soup.find('td', string=re.compile(attribute)).find_next_sibling('td').text.strip()
            if value == "Nein":
                value = False
            elif value == "Ja":
                value = True
            elif value == "keine Information":
                value = None

            if attribute in ["Astbruchgefahr", "Blattform", "Blütenfarbe", "Blütenstand", "Bodeneigenschaft", "Bodenfeuchtetoleranz", "Bodenverdichtungstoleranz", "Fruchtfarbe", "Fruchtform", "Gründigkeit", "Herbstfärbung", "Herkunft", "Hitzeverträglichkeit", "Kronendurchlässigkeit", "Kronenform", "Lichtanspruch", "name_botanic", "name_trivial", "Natürlicher Lebensraum", "Salzverträglichkeit", "Spätfrosttoleranz", "Trockenheitstoleranz", "Wuchsform", "Wuchsgeschwindigkeit", "Wuchsrichtung"] and type(value) == str:
                value = value.split(", ")
                
            tree_information[attribute] = value

    trees.append(tree_information)

100%|██████████| 2/2 [00:00<00:00, 37.72it/s]


## Daten als JSON ablegen

Die eingelesenen Daten wollen wir für eine mögliche Weiterverarbeitung als JSON ablegen. Hierfür können wir das erzeugte Array aus dem vorigen Schritt ablegen.

In [7]:
import json

with open("tree-information.json", "w") as f:
    json.dump({
        "attributes": attributes,
        "trees": trees
    }, f, indent=4)

In [5]:
import pandas as pd

In [2]:
!pip install pandas

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 5.0 MB/s eta 0:00:000:00:0136m0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 4.9 MB/s eta 0:00:00 MB/s eta 0:00:01:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
url = "https://citree.de/AJAX/getSteckbrief.php?ArtInfos=%7B%22ID%22%3A1%2C%22matched%22%3A%5B44%5D%2C%22unmatched%22%3A%5B0%5D%7D&language=de"

url

'https://citree.de/AJAX/getSteckbrief.php?ArtInfos=%7B%22ID%22%3A1%2C%22matched%22%3A%5B44%5D%2C%22unmatched%22%3A%5B0%5D%7D&language=de'

In [7]:
!pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 8.2 MB/s eta 0:00:008.4 MB/s eta 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [15]:
lst_df = pd.read_html(url, encoding='utf-8')

len(lst_df)

11

In [11]:
lst_df[0]

,0,1
0,Lichtanspruch,"sonnig, lichtschattig, halbschattig"
1,Trockenheitstoleranz,trockentolerant
2,HitzevertrÃ¤glichkeit,schwach
3,SpÃ¤tfrosttoleranz,gut
4,WinterhÃ¤rtezone,6b


In [16]:
lst_df[1]

,0,1
0,pH-Wert,5--7.5
1,Bodenverdichtungstoleranz,mittel
2,Staunässetoleranz,kurzfristig tolerant
3,Salzverträglichkeit,gut
4,Bodenfeuchtetoleranz,"feucht, leicht feucht/frisch/gelegentlich troc..."
5,Bodeneigenschaft,"sandig (= leichte Böden), lehmig oder schluffi..."
6,Gründigkeit,"mittel, tief"


In [18]:
len(lst_df)

11

In [ ]:
lst_df[2]


,0,1
0,Mehrstämmigkeit,Nein
1,oberflächennahe Wurzeln,Ja
2,maximale Wuchshöhe,20 m
3,mittlere Wuchshöhe,9 m
4,Kronendurchmesser,5 m
5,Wuchsgeschwindigkeit,langsam bis normal
6,Kronendurchlässigkeit,mittel
7,Wuchsform,Baum
8,Wuchsrichtung,aufrecht oder straff aufrecht
9,Kronenform,"ausgebreitet oder ausladend, eiförmig, rundlich"


In [24]:
from glob import glob

file_list = glob("data/*.html")

In [32]:
def get_df_from_url(url):

    lst_df = pd.read_html(url, encoding='utf-8')

    lst_names = ['Klimabedingungen', 
    'Bodenbedingungen', 
    'Natürliche Verbreitung', 
    'Habitus', 
    'Blatt', 
    'Blüte', 
    'Frucht', 
    'Gefährdung', 
    'Beeinträchtigungen']


    dct_df = {}

    for name, df in zip(lst_names, lst_df):
        dct_df[name] = df.assign(Quelle = url)

    return dct_df

tree_data = []
for file in file_list:
    tree_data.append(get_df_from_url(file))

In [31]:
file


'data/154.html'

In [65]:
lst_names = ['Klimabedingungen', 
    'Bodenbedingungen', 
    'Natürliche Verbreitung', 
    'Habitus', 
    'Blatt', 
    'Blüte', 
    'Frucht', 
    'Gefährdung', 
    'Beeinträchtigungen']


dct_big_df = {}

for name in lst_names:

    df_temp = (
        pd.concat([tree_data[i][name] for i in range(len(tree_data))], ignore_index=True)
        .sort_values(by='Quelle')
        .reset_index(drop=True)
        .rc(['Quelle'])
    )

    dct_big_df[name] = df_temp
    df_temp.to_csv(f"export_{name}.csv", sep=';', index=False)
    

dct_big_df['Bodenbedingungen']



,Quelle,0,1
0,data/1.html,Gründigkeit,"mittel, tief"
1,data/1.html,Bodenfeuchtetoleranz,"feucht, leicht feucht/frisch/gelegentlich troc..."
2,data/1.html,Salzverträglichkeit,gut
3,data/1.html,Staunässetoleranz,kurzfristig tolerant
4,data/1.html,Bodenverdichtungstoleranz,mittel
...,...,...,...
2613,data/99.html,Bodenfeuchtetoleranz,"feucht, leicht feucht/frisch/gelegentlich troc..."
2614,data/99.html,Bodeneigenschaft,"sandig (= leichte Böden), lehmig oder schluffi..."
2615,data/99.html,Gründigkeit,"flach, mittel, tief"
2616,data/99.html,pH-Wert,4--8


In [66]:
@as_method
def ur(DF, bl_print=True):
    """Returns Unique Rows."""

    DF_unique = DF.drop_duplicates()

    if bl_print:
        print(len(DF), "\t| Length of DataFrame")
        print(len(DF_unique), "\t| Length of DataFrame without dupps\n")

    return DF_unique



df_temp

,Quelle,0,1
0,data/1.html,Krankheiten / Schädlinge,"Blattläuse, Wurzelfäule, Verticillium"
1,data/1.html,Pflegeaufwand,mittel
2,data/1.html,Baumunterpflanzung,Ja
3,data/10.html,Baumunterpflanzung,Ja
4,data/10.html,Krankheiten / Schädlinge,"Hallimasch, Phytophthora, Wurzelfäule, Vertici..."
...,...,...,...
1117,data/98.html,Baumunterpflanzung,Ja
1118,data/98.html,Pflegeaufwand,mittel
1119,data/99.html,Krankheiten / Schädlinge,"Rindenkrebs, Raupen, Feuerbrand, Wurzelfäule"
1120,data/99.html,Baumunterpflanzung,Nein


In [68]:
@as_method
def ur(DF, bl_print=True):
    """Returns Unique Rows."""

    DF_unique = DF.drop_duplicates()

    if bl_print:
        print(len(DF), "\t| Length of DataFrame")
        print(len(DF_unique), "\t| Length of DataFrame without dupps\n")

    return DF_unique

df_temp.drop_duplicates()



,Quelle,0,1
0,data/1.html,Krankheiten / Schädlinge,"Blattläuse, Wurzelfäule, Verticillium"
1,data/1.html,Pflegeaufwand,mittel
2,data/1.html,Baumunterpflanzung,Ja
3,data/10.html,Baumunterpflanzung,Ja
4,data/10.html,Krankheiten / Schädlinge,"Hallimasch, Phytophthora, Wurzelfäule, Vertici..."
...,...,...,...
1117,data/98.html,Baumunterpflanzung,Ja
1118,data/98.html,Pflegeaufwand,mittel
1119,data/99.html,Krankheiten / Schädlinge,"Rindenkrebs, Raupen, Feuerbrand, Wurzelfäule"
1120,data/99.html,Baumunterpflanzung,Nein


In [59]:
!ls

citree-scraper.ipynb               export_Gefährdung.csv
data                               export_Habitus.csv
export_Beeinträchtigungen.csv      export_Klimabedingungen.csv
export_Blatt.csv                   export_Natürliche Verbreitung.csv
export_Blüte.csv                   images
export_Bodenbedingungen.csv        tree-information.json
export_Frucht.csv


In [61]:
!ls

citree-scraper.ipynb               export_Gefährdung.csv
data                               export_Habitus.csv
export_Beeinträchtigungen.csv      export_Klimabedingungen.csv
export_Blatt.csv                   export_Natürliche Verbreitung.csv
export_Blüte.csv                   images
export_Bodenbedingungen.csv        tree-information.json
export_Frucht.csv


In [57]:
import appscript

ModuleNotFoundError: No module named 'appscript'

In [55]:
import xlwings

import appscript
import psutil

dct_big_df['Blatt'].view()




ModuleNotFoundError: No module named 'appscript'

In [ ]:
from functools import wraps
from copy import deepcopy
from pandas.core.base import PandasObject

def as_method(func):
    """
    This decrator makes a function also available as a method.
    The first passed argument must be a DataFrame.
    """
    # from functools import wraps
    # from copy import deepcopy
    # import pandas as pd
    # from pandas.core.base import PandasObject

    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*deepcopy(args), **deepcopy(kwargs))

    setattr(PandasObject, wrapper.__name__, wrapper)

    return wrapper

PandasObject.view = xlwings.view

In [45]:
!pip install xlwings


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
for item  in tree_data:
    print(tree_data[i]['Klimabedingungen'])

In [35]:
type(tree_data)



list

In [20]:
dct_df['Bodenbedingungen']


,0,1
0,pH-Wert,5--7.5
1,Bodenverdichtungstoleranz,mittel
2,Staunässetoleranz,kurzfristig tolerant
3,Salzverträglichkeit,gut
4,Bodenfeuchtetoleranz,"feucht, leicht feucht/frisch/gelegentlich troc..."
5,Bodeneigenschaft,"sandig (= leichte Böden), lehmig oder schluffi..."
6,Gründigkeit,"mittel, tief"
